# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which exposes machine-actionable dataset structure and metadata.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`. According to the Croissant schema, entities are referenced by `@id`.

In [ ]:
print("\nRecord Sets available in this dataset:")
for record_set in metadata.record_sets:
    print(f"- Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field name: {field.name}  (@id: {field.id}, dataType: {field.data_type})")
    print("  Columns:")
    for col in record_set.columns:
        print(f"    - Column name: {col.name} (@id: {col.id})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We demonstrate this for the main record set with all referenced by their `@id`.

In [ ]:
# Collect all record set @id
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Record Set @id's in dataset:", record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

# Show columns for main record set
main_record_set_id = record_sets_ids[0] if len(record_sets_ids) > 0 else None
if main_record_set_id:
    print(f"\nColumns for record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Please review the output of the previous section for your available field `@id`s (they directly match DataFrame columns).

In [ ]:
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Example: select a numeric field (edit the field as according to your dataset contents)
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Try defaulting to a common numeric field name, otherwise print
numeric_candidates = [col for col in df.columns if 'abundance' in col.lower() or 'count' in col.lower() or 'intensity' in col.lower() or 'coverage' in col.lower() or df[col].dtype!=object]
if len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]
else:
    # fallback to the first numeric-looking col, else notify
    numeric_field = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_field = numeric_field[0] if len(numeric_field) > 0 else None
if not numeric_field:
    print("No numeric field identified. Please update the field name below.")
    numeric_field = df.columns[0]  # Fallback

print(f"Using numeric field for EDA: {numeric_field}")

# Drop NA for calculations
working_df = df.copy()
working_df[numeric_field] = pd.to_numeric(working_df[numeric_field], errors='coerce')
working_df = working_df.dropna(subset=[numeric_field])

threshold = working_df[numeric_field].mean()
filtered_df = working_df[working_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): Found {len(filtered_df)} records.")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field if present (e.g., 'description' or 'accession')
group_field_candidates = [col for col in df.columns if col not in [numeric_field] and df[col].dtype==object]
group_field = group_field_candidates[0] if len(group_field_candidates) > 0 else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(f"Grouped data by {group_field} (average {numeric_field}):")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Use common libraries like matplotlib and seaborn for efficient plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.histplot(working_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field:
    top_categories = filtered_df[group_field].value_counts().head(6).index
    subset = filtered_df[filtered_df[group_field].isin(top_categories)]
    plt.figure(figsize=(10,6))
    sns.boxplot(data=subset, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} grouped by {group_field} (top 6)")
    plt.xticks(rotation=35)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² Mass Spectrometry Analysis of Extracellular Vesicles data using `mlcroissant`. We:

- Loaded dataset metadata and record sets using their `@id` as required by the Croissant schema.
- Explored the available fields and columns.
- Extracted records and performed basic exploratory data analysis, including filtering, normalization, grouping, and visualization.

For more advanced analysis, investigate the data dictionary and documentation provided in the Croissant schema or dataset repository.
